# 03_base_master — Dataset para Modelagem de Churn

Constrói o dataset de modelagem a partir dos parquets gerados nos notebooks anteriores.

**Estratégia temporal:**
- `CUTOFF_TREINO`: última data usada para features de treino
- Features: comportamento do cliente **antes** do CUTOFF
- Target `churn_h3`: o cliente pediu algo nos `HORIZONTE_MESES` meses após o CUTOFF?
  - `0` → pediu (ativo)
  - `1` → não pediu (churnou)

| Input | Origem | Features |
|---|---|---|
| `cliente_mes.parquet` | notebook 01 | volume, valor, regularidade mensal |
| `cliente_fidelidade.parquet` | notebook 01 | padrão de intervalos entre compras |
| `cliente_item_tendencia.parquet` | notebook 02 | tendência de queda por item |
| Banco | `base_clientes_enr.sql` | NOME, DIASINADIMPLENTE |

**Output:** `data/processed/df_model.parquet`

In [62]:
import sys, os, warnings
from pathlib import Path
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(r"C:\Users\carva\central_eto")
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from src import db
from src.config import (
    DATA_REF, INICIO,
    CUTOFF_TREINO, CUTOFF_TESTE, HORIZONTE_MESES,
    PROCESSED_DIR, QUERIES_DIR,
)

anos = sorted({INICIO[:4], str(CUTOFF_TREINO.year), str(CUTOFF_TESTE.year), str(DATA_REF.year)})
linha = "|"
for ano in anos:
    linha += f"─── {ano} ───|"

ct  = CUTOFF_TREINO.strftime("%d/%m/%y")
cte = CUTOFF_TESTE.strftime("%d/%m/%y")
dr  = DATA_REF.strftime("%d/%m/%y")

print(linha)
print(f"             ↑            ↑             ↑")
print(f"      CUTOFF_TREINO  CUTOFF_TESTE   DATA_REF")
print(f"        ({ct})   ({cte})   ({dr})")
print()
print(f"  Features treino : {INICIO[:7]} → {CUTOFF_TREINO.strftime('%Y-%m')}")
print(f"  Outcome treino  : {(CUTOFF_TREINO + pd.DateOffset(months=1)).strftime('%Y-%m')} → {(CUTOFF_TREINO + pd.DateOffset(months=HORIZONTE_MESES)).strftime('%Y-%m')} ({HORIZONTE_MESES} meses)")
print(f"  Features teste  : {INICIO[:7]} → {CUTOFF_TESTE.strftime('%Y-%m')}")
print(f"  Outcome teste   : {(CUTOFF_TESTE + pd.DateOffset(months=1)).strftime('%Y-%m')} → {(CUTOFF_TESTE + pd.DateOffset(months=HORIZONTE_MESES)).strftime('%Y-%m')} ({HORIZONTE_MESES} meses)")

|─── 2023 ───|─── 2024 ───|─── 2025 ───|─── 2026 ───|
             ↑            ↑             ↑
      CUTOFF_TREINO  CUTOFF_TESTE   DATA_REF
        (31/12/24)   (31/12/25)   (01/05/26)

  Features treino : 2023-01 → 2024-12
  Outcome treino  : 2025-01 → 2025-03 (3 meses)
  Features teste  : 2023-01 → 2025-12
  Outcome teste   : 2026-01 → 2026-03 (3 meses)


---
## Seção 1 — Carregar dados

In [63]:
cm  = pd.read_parquet(PROCESSED_DIR / "cliente_mes.parquet")
cf  = pd.read_parquet(PROCESSED_DIR / "cliente_fidelidade.parquet")
cit = pd.read_parquet(PROCESSED_DIR / "cliente_item_tendencia.parquet")

# ANO_MES pode ser salvo como string em parquet — garantir Period
cm["ANO_MES"] = cm["ANO_MES"].astype(str).pipe(lambda s: pd.PeriodIndex(s, freq="M"))

print(f"cliente_mes            : {cm.shape}")
print(f"cliente_fidelidade     : {cf.shape}")
print(f"cliente_item_tendencia : {cit.shape}")

cliente_mes            : (9916, 9)
cliente_fidelidade     : (764, 15)
cliente_item_tendencia : (12967, 9)


In [64]:
sql = Path(QUERIES_DIR / "base_clientes.sql").read_text()
df_clientes_raw = db.get_data(sql)

print("Colunas disponíveis:", list(df_clientes_raw.columns))

Colunas disponíveis: ['CODIGO', 'NOME', 'CLASSE', 'CIDADEENTREGA', 'UFENTREGA', 'ATIVIDADE', 'STATUS', 'CODCONDICAOPAGAMENTO', 'CONDICAOPAGAMENTO', 'DIASINADIMPLENTE']


In [65]:
df_perfil = (
    df_clientes_raw[["CODIGO", "NOME", "DIASINADIMPLENTE"]]
    .drop_duplicates(subset="CODIGO")
    .rename(columns={"CODIGO": "CLIENTE"})
)

print(f"Perfil de clientes: {df_perfil.shape}")

Perfil de clientes: (1148, 3)


---
## Seção 2 — Janela temporal e target

Definimos o CUTOFF como fim da janela de features. O target é calculado a partir
dos pedidos na janela de outcome (`HORIZONTE_MESES` meses após o CUTOFF).

| Label | Significado |
|---|---|
| `churn_h3 = 0` | cliente pediu na janela de outcome — ativo |
| `churn_h3 = 1` | cliente NÃO pediu na janela de outcome — churnou |

In [66]:
cutoff_period = pd.Period(CUTOFF_TREINO, "M")          # 2024-12
outcome_end   = cutoff_period + HORIZONTE_MESES         # 2025-03

# Janela de features: meses ATÉ o CUTOFF (inclusive)
cm_feat    = cm[cm["ANO_MES"] <= cutoff_period].copy()

# Janela de outcome: meses APÓS o CUTOFF até o fim do horizonte
cm_outcome = cm[
    (cm["ANO_MES"] > cutoff_period) &
    (cm["ANO_MES"] <= outcome_end)
].copy()

# Clientes elegíveis: tiveram pelo menos 1 pedido antes do CUTOFF
clientes_elegiveis = cm_feat["CLIENTE"].unique()

# Target
comprou_pos = set(cm_outcome["CLIENTE"].unique())
df_target = pd.DataFrame({"CLIENTE": clientes_elegiveis})
df_target["churn_h3"] = (~df_target["CLIENTE"].isin(comprou_pos)).astype(int)

print(f"Janela features: {INICIO} → {cutoff_period}")
print(f"Janela outcome : {cutoff_period + 1} → {outcome_end} ({HORIZONTE_MESES} meses)")
print(f"")
print(f"Clientes elegíveis: {len(df_target)}")
print(f"Churn  (churn_h3=1): {df_target['churn_h3'].sum()} ({df_target['churn_h3'].mean():.1%})")
print(f"Ativos (churn_h3=0): {(df_target['churn_h3']==0).sum()} ({(df_target['churn_h3']==0).mean():.1%})")

Janela features: 2023-01-01 → 2024-12
Janela outcome : 2025-01 → 2025-03 (3 meses)

Clientes elegíveis: 610
Churn  (churn_h3=1): 260 (42.6%)
Ativos (churn_h3=0): 350 (57.4%)


---
## Seção 3 — Features de comportamento mensal

Agregamos `cliente_mes` no nível cliente usando **apenas a janela de features** (pré-CUTOFF).

| Feature | Descrição |
|---|---|
| `n_meses_ativos` | Meses com pelo menos 1 pedido na janela |
| `total_pedidos` | Total de pedidos únicos na janela |
| `total_valor` | Faturamento total na janela |
| `media_pedidos_mes` | Média de pedidos por mês |
| `cv_pedidos` | Coeficiente de variação — regularidade (maior = mais irregular) |
| `ticket_medio` | Ticket médio na janela |
| `itens_por_pedido` | Média de itens por pedido |
| `categoria_pedido` | Categoria predominante (moda) |
| `meses_sem_pedido_pre` | Meses de silêncio até o CUTOFF |

In [67]:
def meses_desde(serie_periodo: pd.Series, ref: pd.Period) -> pd.Series:
    return serie_periodo.apply(
        lambda x: (ref.year - x.year) * 12 + (ref.month - x.month)
    )

# Janela total de meses disponível (INICIO até CUTOFF)
janela_meses = (
    (cutoff_period.year - pd.Period(INICIO, "M").year) * 12 +
    (cutoff_period.month - pd.Period(INICIO, "M").month)
)

features_comportamento = (
    cm_feat.groupby("CLIENTE")
    .agg(
        n_meses_ativos    = ("ANO_MES",         "count"),
        ultimo_mes_pre    = ("ANO_MES",          "max"),
        total_pedidos     = ("total_pedidos",    "sum"),
        total_valor       = ("total_valor",      "sum"),
        media_pedidos_mes = ("total_pedidos",    "mean"),
        std_pedidos_mes   = ("total_pedidos",    "std"),
        ticket_medio      = ("ticket_medio",     "mean"),
        itens_por_pedido  = ("itens_por_pedido", "mean"),
        categoria_pedido  = ("categoria_pedido", lambda x: x.mode()[0]),
    )
    .reset_index()
)

features_comportamento["cv_pedidos"] = (
    features_comportamento["std_pedidos_mes"] /
    features_comportamento["media_pedidos_mes"]
).fillna(0)

features_comportamento["meses_sem_pedido_pre"] = meses_desde(
    features_comportamento["ultimo_mes_pre"], cutoff_period
)

# razao_atividade: proporção de meses ativos na janela
# substitui a bifurcação baixo_recorrente/sazonal — o modelo aprende o limiar
features_comportamento["razao_atividade"] = (
    features_comportamento["n_meses_ativos"] / janela_meses
)

features_comportamento = features_comportamento.drop(
    columns=["ultimo_mes_pre", "std_pedidos_mes"]
)

print(f"Janela: {janela_meses} meses ({INICIO} → {cutoff_period})")
print(f"Shape: {features_comportamento.shape}")
display(features_comportamento.describe().round(2))

Janela: 23 meses (2023-01-01 → 2024-12)
Shape: (610, 11)


,CLIENTE,n_meses_ativos,total_pedidos,total_valor,media_pedidos_mes,ticket_medio,itens_por_pedido,cv_pedidos,meses_sem_pedido_pre,razao_atividade
count,610.00,610.00,610.00,610.00,610.00,610.00,610.00,610.00,610.00,610.00
mean,635.67,9.07,79.59,16642.43,4.48,197.23,3.66,0.26,4.59,0.39
std,430.37,8.29,525.97,95830.49,22.02,225.88,2.91,0.25,6.10,0.36
min,1.00,1.00,1.00,0.00,1.00,0.00,1.00,0.00,0.00,0.04
25%,224.25,2.00,2.00,239.00,1.00,71.78,1.50,0.00,0.00,0.09
50%,565.00,6.00,8.00,1011.20,1.33,129.67,2.80,0.30,2.00,0.26
75%,1055.75,15.00,31.00,5117.48,2.32,246.37,5.00,0.45,8.00,0.65
max,1372.00,24.00,10698.00,1639520.45,445.75,2289.54,22.00,1.16,23.00,1.04


---
## Seção 4 — Features de fidelidade

Padrão histórico de intervalos entre compras — comportamento estrutural do cliente.

> Features **excluídas por leakage**: `meses_sumido`, `RED_FLAG`, `valor_ultimo_mes`,
> `threshold_churn`, `historico_confiavel` — todas dependem de `DATA_REF`.

In [68]:
COLS_FIDELIDADE = [
    "CLIENTE",
    "intervalo_medio",   # média de dias entre pedidos consecutivos
    "max_intervalo",     # maior intervalo registrado
    "n_intervalos",      # quantidade de intervalos (proxy de histórico)
    "categoria_cliente", # perfil geral (moda de categoria_pedido)
]

features_fidelidade = cf[COLS_FIDELIDADE].copy()

print(f"Shape: {features_fidelidade.shape}")
display(features_fidelidade.describe().round(2))

Shape: (764, 5)


,CLIENTE,intervalo_medio,max_intervalo,n_intervalos
count,764.00,625.00,764.00,764.00
mean,788.04,3.52,4.66,11.98
std,513.85,4.61,5.79,13.03
min,1.00,1.00,0.00,0.00
25%,291.25,1.04,1.00,1.00
50%,801.50,1.58,2.00,6.00
75%,1248.25,3.67,6.00,21.00
max,1674.00,29.00,33.00,40.00


---
## Seção 5 — Features de tendência de itens

Captura o sinal de migração gradual para concorrente: queda progressiva no volume
de itens específicos antes do churn.

> **Leakage residual:** `tendencia_slope` foi calculado sobre todo o período (2023–DATA_REF).
> Influência dos meses pós-CUTOFF é pequena (regressão linear sobre 2+ anos).
> `var_pct_ultimo` foi **excluído** — representa o último mês dos dados (muito posterior ao CUTOFF).

In [69]:
features_itens = (
    cit.groupby("CLIENTE")
    .agg(
        slope_portfolio_medio = ("tendencia_slope", "mean"),
        pct_itens_queda       = ("tendencia_slope", lambda x: (x < 0).sum() / len(x)),
        n_itens_portfolio     = ("SERVICO",         "count"),
    )
    .reset_index()
)

print(f"Shape: {features_itens.shape}")
display(features_itens.describe().round(3))

Shape: (764, 4)


,CLIENTE,slope_portfolio_medio,pct_itens_queda,n_itens_portfolio
count,764.000,602.000,764.000,764.000
mean,788.045,-0.025,0.350,16.973
std,513.853,0.292,0.287,29.790
min,1.000,-4.000,0.000,1.000
25%,291.250,-0.036,0.000,3.000
50%,801.500,-0.000,0.374,9.000
75%,1248.250,0.019,0.500,18.000
max,1674.000,2.000,1.000,512.000


---
## Seção 6 — Montar df_model

Join sequencial partindo do target (clientes elegíveis como âncora).

In [70]:
df_model = (
    df_target
    .merge(features_comportamento, on="CLIENTE", how="left")
    .merge(features_fidelidade,    on="CLIENTE", how="left")
    .merge(features_itens,         on="CLIENTE", how="left")
    .merge(df_perfil,              on="CLIENTE", how="left")
)

print(f"Shape final: {df_model.shape}")
print(f"Colunas    : {list(df_model.columns)}")

missing = df_model.isna().sum()
missing = missing[missing > 0]
if len(missing):
    print(f"\nMissing values:")
    print(missing)
else:
    print("\nSem missing values.")

Shape final: (610, 21)
Colunas    : ['CLIENTE', 'churn_h3', 'n_meses_ativos', 'total_pedidos', 'total_valor', 'media_pedidos_mes', 'ticket_medio', 'itens_por_pedido', 'categoria_pedido', 'cv_pedidos', 'meses_sem_pedido_pre', 'razao_atividade', 'intervalo_medio', 'max_intervalo', 'n_intervalos', 'categoria_cliente', 'slope_portfolio_medio', 'pct_itens_queda', 'n_itens_portfolio', 'NOME', 'DIASINADIMPLENTE']

Missing values:
intervalo_medio           85
slope_portfolio_medio    104
dtype: int64


In [71]:
print("=== Target ===")
vc = df_model["churn_h3"].value_counts().rename({0: "Ativo", 1: "Churnou"})
print(vc)
print(f"Taxa de churn: {df_model['churn_h3'].mean():.1%}")

print("\n=== Por categoria ===")
print(
    df_model.groupby("categoria_cliente")["churn_h3"]
    .agg(n="count", churnou="sum", taxa="mean")
    .assign(taxa=lambda x: x["taxa"].map("{:.1%}".format))
)

=== Target ===
churn_h3
Ativo      350
Churnou    260
Name: count, dtype: int64
Taxa de churn: 42.6%

=== Por categoria ===
                     n  churnou   taxa
categoria_cliente                     
alto                58       12  20.7%
baixo              493      240  48.7%
medio               44        5  11.4%
premium             15        3  20.0%


In [72]:
# intervalo_medio: NaN = cliente com só 1 mês de compra (sem intervalo calculável)
# Preenchemos com o dobro do max observado — sinaliza "intervalo grande/desconhecido"
intervalo_max = df_model["intervalo_medio"].max()
df_model["intervalo_medio"] = df_model["intervalo_medio"].fillna(intervalo_max * 2)

# slope e pct_itens_queda: NaN = sem histórico de itens suficiente para calcular tendência
# Preenchemos com 0 — neutro, sem sinal detectável
df_model["slope_portfolio_medio"] = df_model["slope_portfolio_medio"].fillna(0)
df_model["pct_itens_queda"]       = df_model["pct_itens_queda"].fillna(0)

missing = df_model.isna().sum()
missing = missing[missing > 0]
if len(missing):
    print("Missing restantes:")
    print(missing)
else:
    print("Sem missing values.")

Sem missing values.


---
## Seção 7 — Salvar

Output: `data/processed/df_model.parquet` — input do `04_modelo.ipynb`.

In [73]:
df_model.to_parquet(PROCESSED_DIR / "df_model.parquet", index=False)

print("=== Salvo ===")
print(f"  {PROCESSED_DIR / 'df_model.parquet'}")
print(f"  Shape  : {df_model.shape}")
print(f"  Colunas: {list(df_model.columns)}")
print(f"  Churn  : {df_model['churn_h3'].sum()} ({df_model['churn_h3'].mean():.1%})")

=== Salvo ===
  C:\Users\carva\central_eto\data\processed\df_model.parquet
  Shape  : (610, 21)
  Colunas: ['CLIENTE', 'churn_h3', 'n_meses_ativos', 'total_pedidos', 'total_valor', 'media_pedidos_mes', 'ticket_medio', 'itens_por_pedido', 'categoria_pedido', 'cv_pedidos', 'meses_sem_pedido_pre', 'razao_atividade', 'intervalo_medio', 'max_intervalo', 'n_intervalos', 'categoria_cliente', 'slope_portfolio_medio', 'pct_itens_queda', 'n_itens_portfolio', 'NOME', 'DIASINADIMPLENTE']
  Churn  : 260 (42.6%)
